# ATP-safe expansion of the FSP237 curated CMM into a GSM

**Strategy:** start with the curated central-carbon model (CMM, KBase `169876/170/5`) which gives
exactly 30 ATP/mmol glucose aerobically and 2 anaerobically. Iteratively add reactions from the
larger fsp237 GSM (`fsp237_minimal_glucose.json`, post-ETC-fixes), rejecting any reaction that lets
the model exceed the curated ATP yield -- those are introduced futile cycles.

**CMM precedence:** for reactions that exist in both CMM and GSM with different bounds/stoichiometry,
keep the CMM version by default. Only swap to the GSM version if biomass cannot grow otherwise --
then find the **minimum** swap set.

**Workflow:**
1. Load CMM (cached locally; fetched once from KBase)
2. Verify baseline ATP yields (30 aerobic, 2 anaerobic)
3. Load source GSM, identify GSM-unique reactions
4. ATP-safe expansion (batched + bisection)
5. Add the GSM biomass reaction
6. Test biomass on minimal-glucose media
7. If biomass = 0, find minimum CMM->GSM shared-rxn swap set
8. If still 0 with all swaps, do the secondary re-introduction from blacklist
9. Save deliverables

Heavy lifting is in `atp_safe_expand.py`.

In [ ]:
import os, json
import cobra
import pandas as pd

%load_ext autoreload
%autoreload 2
from atp_safe_expand import (
    atp_yield, gsm_unique_reactions, expand_atp_safe,
    add_reactions_from, test_biomass,
    find_mismatched_shared, minimum_shared_swaps_for_biomass,
    minimum_blacklist_for_biomass,
)

BASE = '/home/janakae/fungalTemplate/imm904CobraModel'
OUT  = f'{BASE}/fsp237_atp_safe_gsm'
CMM_JSON = f'{OUT}/cmm_169876_170_5.json'
GSM_JSON = f'{BASE}/fsp237_minimal_glucose/fsp237_minimal_glucose.json'
FINAL_MODEL = f'{OUT}/fsp237_atp_safe_gsm.json'

print(f'output dir: {OUT}')

## 1. Load CMM (fetch from KBase only if cache missing)

In [ ]:
if not os.path.exists(CMM_JSON):
    print('cache miss; fetching from KBase ...')
    import cobrakbase
    kbase = cobrakbase.KBaseAPI()
    cmm_kb = kbase.get_from_ws('169876/170/5')
    cobra.io.save_json_model(cmm_kb, CMM_JSON)

cmm = cobra.io.load_json_model(CMM_JSON)
print(f'CMM: {cmm.id}')
print(f'  reactions: {len(cmm.reactions)}, metabolites: {len(cmm.metabolites)}')
ATP_DEMAND_RXN = 'bio1'   # in the CMM, bio1 is ATP+H2O->ADP+Pi+H+

## 2. Verify baseline ATP yields

In [ ]:
baseline = atp_yield(cmm, ATP_DEMAND_RXN, glc_uptake=1.0)
print(f'CMM baseline:')
print(f'  aerobic   ATP/glucose = {baseline[0]:.4f}   (expected 30)')
print(f'  anaerobic ATP/glucose = {baseline[1]:.4f}   (expected  2)')
assert abs(baseline[0] - 30) < 0.01
assert abs(baseline[1] -  2) < 0.01

## 3. Load source GSM and identify candidates

In [ ]:
gsm = cobra.io.load_json_model(GSM_JSON)
print(f'GSM: {len(gsm.reactions)} reactions, {len(gsm.metabolites)} metabolites')

candidates = gsm_unique_reactions(cmm, gsm)
# Don't add the GSM bio1 as a candidate -- it would conflict with the CMM's ATP-demand bio1.
# We add the real biomass reaction separately, under a new id, after expansion.
if 'bio1' in candidates:
    candidates.remove('bio1')
print(f'GSM-unique reactions to consider: {len(candidates)}')

mismatched = find_mismatched_shared(cmm, gsm)
print(f'shared rxns with bounds/stoich mismatch (left as CMM unless biomass requires swap): '
      f'{len(mismatched)}')

## 4. ATP-safe iterative expansion

Adds in batches of 100; on a batch that pushes ATP yield above baseline, bisects to find the
offenders. Tolerance 1e-3.

In [ ]:
result = expand_atp_safe(
    cmm, gsm, atp_rxn=ATP_DEMAND_RXN,
    candidates=candidates,
    batch_size=100, tol=1e-3, glc_uptake=1.0, verbose=True,
)
print(f'\nsafely added: {len(result.safe_added)}')
print(f'blacklisted : {len(result.blacklisted)} -> {result.blacklisted}')
print(f'final ATP yields: aerobic={result.final_yields[0]:.3f}, anaerobic={result.final_yields[1]:.3f}')

## 5. Add the GSM biomass reaction (as `bio_gsm`)

We keep the CMM's `bio1` (ATP demand) as the ATP-yield objective. Real biomass uses a new id.

In [ ]:
gsm_bio = gsm.reactions.get_by_id('bio1')
missing = [m.id for m in gsm_bio.metabolites if m.id not in [x.id for x in cmm.metabolites]]
if missing:
    cmm.add_metabolites([cobra.Metabolite(id=mid,
                          name=gsm.metabolites.get_by_id(mid).name,
                          formula=gsm.metabolites.get_by_id(mid).formula,
                          charge=gsm.metabolites.get_by_id(mid).charge,
                          compartment=gsm.metabolites.get_by_id(mid).compartment)
                          for mid in missing])
if 'bio_gsm' not in [r.id for r in cmm.reactions]:
    bio_gsm = cobra.Reaction('bio_gsm', name='biomass [from GSM]',
                              lower_bound=0, upper_bound=1000)
    bio_gsm.add_metabolites({cmm.metabolites.get_by_id(m.id): c
                              for m, c in gsm_bio.metabolites.items()})
    cmm.add_reactions([bio_gsm])
print(f'bio_gsm has {len(cmm.reactions.get_by_id("bio_gsm").metabolites)} metabolites')

## 6. Find the minimum set of shared-rxn swaps needed for biomass

Respects CMM precedence: tries with NO swaps first; only swaps if biomass = 0; then finds the
minimum subset of mismatched shared reactions to swap.

In [ ]:
swapped, kept_cmm = minimum_shared_swaps_for_biomass(
    cmm, gsm, biomass_rxn='bio_gsm', atp_rxn=ATP_DEMAND_RXN,
    baseline_yields=baseline, glc_uptake_atp=1.0, glc_uptake_bio=5.0,
    use_gsm_media=True, biomass_threshold=1e-6, atp_tol=1e-3, verbose=True,
)

## 7. Final verification: biomass on minimal-glucose media + ATP yields

In [ ]:
# Apply GSM-style media
gsm_bounds = {r.id: (r.lower_bound, r.upper_bound) for r in gsm.exchanges}
for ex in cmm.exchanges:
    if ex.id in gsm_bounds:
        ex.lower_bound, ex.upper_bound = gsm_bounds[ex.id]
    else:
        ex.lower_bound = 0

# Aerobic biomass
for r in cmm.reactions: r.objective_coefficient = 1 if r.id == 'bio_gsm' else 0
sol_a = cmm.optimize()
# Anaerobic biomass
cmm.reactions.get_by_id('EX_cpd00007_e0').lower_bound = 0
sol_n = cmm.optimize()
cmm.reactions.get_by_id('EX_cpd00007_e0').lower_bound = -10

yf = atp_yield(cmm, ATP_DEMAND_RXN, glc_uptake=1.0)
print(f'FINAL biomass aerobic   = {sol_a.objective_value:.4f}')
print(f'FINAL biomass anaerobic = {sol_n.objective_value:.4f}')
print(f'FINAL ATP yields        = aerobic {yf[0]:.3f}, anaerobic {yf[1]:.3f}')
print(f'reactions: {len(cmm.reactions)}, metabolites: {len(cmm.metabolites)}')

## 8. Secondary loop: if biomass STILL = 0 after all swaps, re-introduce from blacklist

Skipped here because biomass already > 0. Runs only if the model couldn't grow with any combination
of allowed shared-rxn swaps.

In [ ]:
reintroduced = []
if sol_a.objective_value is None or sol_a.objective_value < 1e-6:
    print('biomass still 0; running blacklist re-introduction...')
    reintroduced = minimum_blacklist_for_biomass(
        cmm, gsm, blacklist=result.blacklisted,
        biomass_rxn='bio_gsm', atp_rxn=ATP_DEMAND_RXN,
        baseline_yields=baseline, glc_uptake=1.0, verbose=True,
    )
    sol_a = cmm.optimize()
    print(f'after re-intro of {len(reintroduced)} blacklisted: biomass={sol_a.objective_value:.4f}')
else:
    print('biomass grows -- no re-introduction needed.')

## 9. Save deliverables

In [ ]:
cobra.io.save_json_model(cmm, FINAL_MODEL)

def rxn_row(rid, src, why):
    r = src.reactions.get_by_id(rid)
    comp = sorted({m.id.rsplit('_',1)[-1] for m in r.metabolites})[0] if r.metabolites else ''
    return {'rxn_id': rid, 'name': r.name, 'compartment': comp,
            'bounds': f'({r.lower_bound}, {r.upper_bound})',
            'equation_cpd': r.build_reaction_string(use_metabolite_names=False),
            'gpr': r.gene_reaction_rule, 'subsystem': r.subsystem, 'why': why}

pd.DataFrame([rxn_row(rid, gsm, 'safe addition: ATP yield preserved')
              for rid in result.safe_added]).to_csv(f'{OUT}/safe_added.tsv', sep='\t', index=False)
pd.DataFrame([rxn_row(rid, gsm, 'blacklisted: introduced futile ATP cycle')
              for rid in result.blacklisted]).to_csv(f'{OUT}/blacklist.tsv', sep='\t', index=False)
pd.DataFrame([rxn_row(rid, gsm, 'shared rxn: GSM version replaces CMM for biomass')
              for rid in swapped]).to_csv(f'{OUT}/shared_rxns_swapped.tsv', sep='\t', index=False)
pd.DataFrame([rxn_row(rid, gsm, 'reintroduced from blacklist for biomass')
              for rid in reintroduced]).to_csv(f'{OUT}/reintroduced_for_biomass.tsv', sep='\t', index=False)

print('wrote: fsp237_atp_safe_gsm.json, safe_added.tsv, blacklist.tsv,')
print('       shared_rxns_swapped.tsv, reintroduced_for_biomass.tsv')
print('See REPORT.md for the narrative summary.')